# Cell Type Identification

This notebook assigns cell types to segmented cells using known
pancreatic cell type markers. It uses both a marker-based scoring
approach and unsupervised clustering with annotation.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cpsam_xenium_analysis import config
from cpsam_xenium_analysis.analysis import CellTyper
from cpsam_xenium_analysis.visualization import plots as vis
from cpsam_xenium_analysis.integration.transcript_mapping import save_expression_npz

%matplotlib inline


In [ ]:
# Load expression data for morphology-based segmentation
gene_names = np.load(config.OUTPUT_DIR / 'expr_cpsam_morphology_gene_names.npy', allow_pickle=True).tolist()
cell_labels = np.load(config.OUTPUT_DIR / 'expr_cpsam_morphology_cell_ids.npy')
from scipy.sparse import load_npz
expr_mat = load_npz(str(config.OUTPUT_DIR / 'expr_cpsam_morphology_matrix.npz'))
mask_morph = np.load(config.OUTPUT_DIR / 'masks_cpsam_morphology.npy')

print(f'Expression matrix: {expr_mat.shape}')
print(f'Gene panel: {len(gene_names)} genes')
print(f'Cell mask labels: {len(cell_labels)} unique cells')


## 1. Marker-Based Cell Typing


In [ ]:
typer = CellTyper()
print('Pancreatic cell type markers:')
for ct, genes in typer.markers.items():
    available = [g for g in genes if g in gene_names]
    print(f'  {ct}: {available}')


In [ ]:
cell_types = typer.score_cell_types(expr_mat, gene_names)
print(cell_types['cell_type'].value_counts())
cell_types.head()


In [ ]:
# Visualize cell type map
fig = vis.plot_cell_type_map(
    mask_morph, cell_types, cell_labels,
    title='Cell Type Map (Morphology-based segmentation)'
)


## 2. Marker Gene Expression per Cell Type


In [ ]:
# Expression of key markers across cell types
marker_genes = ['GCG', 'INS', 'SST', 'PRSS1', 'KRT19', 'PECAM1', 'PTPRC']
marker_genes = [g for g in marker_genes if g in gene_names]

type_order = cell_types.groupby('cell_type')['cell_id'].count().sort_values(ascending=False).index

fig, axes = plt.subplots(1, len(marker_genes), figsize=(4*len(marker_genes), 4))
if len(marker_genes) == 1:
    axes = [axes]

for i, gene in enumerate(marker_genes):
    gene_idx = gene_names.index(gene)
    gene_expr = expr_mat[:, gene_idx].toarray().flatten()
    data = pd.DataFrame({'cell_type': cell_types['cell_type'], 'expression': gene_expr})
    means = data.groupby('cell_type')['expression'].mean().reindex(type_order)
    axes[i].bar(range(len(means)), means.values)
    axes[i].set_xticks(range(len(means)))
    axes[i].set_xticklabels(means.index, rotation=45, ha='right', fontsize=6)
    axes[i].set_title(f'{gene}')
    axes[i].set_ylabel('Mean expression')

plt.tight_layout()
plt.show()


## 3. Cluster-Based Cell Typing (requires scanpy)


In [ ]:
# Unsupervised clustering for validation
try:
    cluster_types = typer.cluster_and_annotate(expr_mat, gene_names)
    print('Cluster-based annotation complete')
    
    # Compare with marker-based annotation
    agreement = (cluster_types['cell_type'] == cell_types['cell_type']).mean()
    print(f'Agreement between marker-based and cluster-based: {agreement:.1%}')
except Exception as e:
    print(f'Clustering skipped (install scanpy if desired): {e}')


## 4. H&E-based Cell Typing (if available)


In [ ]:
if (config.OUTPUT_DIR / 'expr_cpsam_he_matrix.npz').exists():
    cell_labels_he = np.load(config.OUTPUT_DIR / 'expr_cpsam_he_cell_ids.npy')
    expr_he = load_npz(str(config.OUTPUT_DIR / 'expr_cpsam_he_matrix.npz'))
    mask_he = np.load(config.OUTPUT_DIR / 'masks_cpsam_he.npy')
    cell_types_he = typer.score_cell_types(expr_he, gene_names)
    
    print('\nH&E cell type distribution:')
    print(cell_types_he['cell_type'].value_counts())
else:
    print('H&E expression data not found. Run transcript mapping first.')
